## Phase 3: Candidate Classification (Part A)

Trains classifiers on the labeled set (CONFIRMED vs FALSE POSITIVE) using group-aware cross-validation (grouped by `kepid` to prevent host-star leakage), accross three tiers to test how much of the accuracy is driven by NASA's own vetting-pipeline outputs vs. genuine physical signal:

1. **Full** - all features, including `koi_score` and `koi_fpflag_*`
2. **No-score** - drop `koi_score` only
3. **Raw-physical** - drop `koi_score` AND all `koi_fpflag_*` columns

Models: Random Forest, XGBoost, Gradient Boosting, SVM, Logisitic Regression, and a MLP Model

Final step: Apply the best full-tier model to the held out CANDIDATE rows to predict which are likely real planets

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras import layers, models
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
tf.random.set_seed(42)

In [5]:
koi_labeled = pd.read_csv("../data/processed/koi_labeled.csv")
koi_candidates  = pd.read_csv("../data/processed/koi_candidates.csv")

print("Labeled shape:", koi_labeled.shape)
print("Candidates shape:", koi_candidates.shape)

Labeled shape: (7587, 112)
Candidates shape: (1977, 111)


### Define the Three Feature Tiers

In [7]:
fpflag_cols = ["koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec"]
non_feature_cols = ["kepid", "koi_disposition", "koi_pdisposition", "target"]

all_features = [c for c in koi_labeled.columns if c not in non_feature_cols]

tier_full = all_features
tier_no_score = [c for c in all_features if c != "koi_score"]
tier_raw_physical = [c for c in all_features if c != "koi_score" and c not in fpflag_cols]

tiers = {
    "Full": tier_full,
    "No-score": tier_no_score,
    "Raw physical": tier_raw_physical
}

for name, cols in tiers.items():
    print(f"{name}: {len(cols)} features")

Full: 108 features
No-score: 107 features
Raw physical: 103 features


### Group-Aware Train/Test Split Setup

Using `GroupKFold` grouped by `kepid` so no host star appears in both training and validation folds within the same split.

In [8]:
X_full = koi_labeled[all_features].copy()
y = koi_labeled["target"].values
groups = koi_labeled["kepid"].values

gkf = GroupKFold(n_splits=5)
splits = list(gkf.split(X_full, y, groups=groups))
print(f"Created {len(splits)} group-aware folds")

Created 5 group-aware folds


### Model Definitions

Includes a helper to build and train the MLP deep learning model per fold.

In [9]:
def build_mlp(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer="ada,", loss="binary_crossentropy", metrics=["accuracy"])
    return model

def get_models():
    return{
        "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
        "XGBoost": xgb.XGBClassifier(n_estimators=300, eval_metric="logloss", random_state=42, n_jobs=-1),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, random_state=42),
        "SVM": SVC(probability=True, random_state=42),
        "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42)
    }

### Cross-Validated Evaluation Across All Tiers and Models

For each tier: scale features, run 5-fold group-aware CV for each sklearn/XGBoost model, then separately train and evaluate the MLP (needs its own loop since Keras models aren't scikit-learn-compatible out of the box).

In [10]:
results = []

for tier_name, tier_cols in tiers.items():
    X = koi_labeled[tier_cols].values

    for model_name in ["Random Forest", "XGBoost", "Gradient Boosting", "SVM", "Logistic Regression"]:
        fold_f1, fold_auc = [], []

        for train_idx, val_idx in splits:
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_val_s = scaler.transform(X_val)

            model = get_models()[model_name]
            model.fit(X_train_s, y_train)
            preds = model.predict(X_val_s)
            proba = model.predict_proba(X_val_s)[:, 1]

            fold_f1.append(f1_score(y_val, preds))
            fold_auc.append(roc_auc_score(y_val, proba))

        results.append({
            "Tier": tier_name,
            "Model": model_name,
            "F1_mean": np.mean(fold_f1),
            "F1_std": np.std(fold_f1),
            "AUC_mean": np.mean(fold_auc),
            "AUC_std": np.std(fold_auc)
        })

        print(f"{tier_name} | {model_name}: F1={np.mean(fold_f1):.4f} AUC={np.mean(fold_auc):.4f}")

Full | Random Forest: F1=0.9884 AUC=0.9993
Full | XGBoost: F1=0.9932 AUC=0.9996
Full | Gradient Boosting: F1=0.9938 AUC=0.9990
Full | SVM: F1=0.9835 AUC=0.9987
Full | Logistic Regression: F1=0.9810 AUC=0.9975
No-score | Random Forest: F1=0.9862 AUC=0.9990
No-score | XGBoost: F1=0.9955 AUC=0.9992
No-score | Gradient Boosting: F1=0.9949 AUC=0.9989
No-score | SVM: F1=0.9702 AUC=0.9967
No-score | Logistic Regression: F1=0.9695 AUC=0.9952
Raw physical | Random Forest: F1=0.9631 AUC=0.9957
Raw physical | XGBoost: F1=0.9724 AUC=0.9967
Raw physical | Gradient Boosting: F1=0.9687 AUC=0.9963
Raw physical | SVM: F1=0.9342 AUC=0.9889
Raw physical | Logistic Regression: F1=0.9367 AUC=0.9890


## Deep Learning Model (MLP) — Separate CV Loop

In [ ]:
for tier_name, tier_cols in tiers.items():
    X = koi_labeled[tier_cols].values
    fold_f1 , fold_auc = [], []

    for train_idx, val_idx in splits:
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_val_s = scaler.transform(X_val)

        model = build_mlp(X_train_s.shape[1])
        model.fit(X_train_s, y_train, epochs=30, batch_size=32, verbose=0,
                  validation_data=(X_val_s, y_val))

        proba = model.predict(X_val_s, verbose=0).flatten()
        preds = (proba > 0.05).astype(int)

        fold_f1.append(f1_score(y_val, preds))
        fold_auc.append(roc_auc_score(y_val, proba))

    results.append({
        "Tier": tier_name,
        "Model": "MLP (Deep Learning)",
        "F1_Mean": np.mean(fold_f1),
        "F1_std": np.std(fold_f1),
        "AUC_mean": np.mean(fold_auc),
        "AUC_std": np.std(fold_auc)
    })
    print(f"{tier_name} | MLP: F1={np.mean(fold_f1):.4f} AUC={np.mean(fold_auc):.4f}")